<a href="https://colab.research.google.com/github/keksenia/cstati-event-analytics/blob/main/notebooks/05_recommendations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 — Product Recommendations

Цель ноутбука — собрать финальные выводы по event portfolio cstati и сформулировать продуктовые рекомендации.

Этот ноутбук отвечает на вопросы:

- какие события работают на рост аудитории;
- какие события лучше удерживают участников;
- где есть сильные event journeys;
- какие события требуют улучшения сбора данных;
- какие действия стоит предпринять организаторам cstati;
- какие выводы стоит вынести в README проекта.

Фокус: сделать не просто аналитический отчёт, а decision-making документ в стиле product analytics.


In [5]:
from pathlib import Path
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

PROJECT_ROOT = Path("/content")

PROCESSED_PUBLIC_DIR = PROJECT_ROOT / "data" / "processed_public"
FIGURES_DIR = PROJECT_ROOT / "docs" / "figures"
DOCS_DIR = PROJECT_ROOT / "docs"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print("PROCESSED_PUBLIC_DIR:", PROCESSED_PUBLIC_DIR)
print("FIGURES_DIR:", FIGURES_DIR)
print("DOCS_DIR:", DOCS_DIR)


PROCESSED_PUBLIC_DIR: /content/data/processed_public
FIGURES_DIR: /content/docs/figures
DOCS_DIR: /content/docs


In [6]:
def load_processed_table(name: str) -> pd.DataFrame:
    parquet_path = PROCESSED_PUBLIC_DIR / f"{name}.parquet"
    csv_path = PROCESSED_PUBLIC_DIR / f"{name}.csv"

    if parquet_path.exists():
        return pd.read_parquet(parquet_path)

    if csv_path.exists():
        return pd.read_csv(csv_path)

    raise FileNotFoundError(
        f"Не найден файл {name}.parquet или {name}.csv в {PROCESSED_PUBLIC_DIR}"
    )


required_tables = [
    "metrics_event_level",
    "metrics_family_level",
    "metrics_repeat_depth",
    "metrics_first_event_retention",
    "event_transitions",
    "family_transitions",
    "event_scorecard",
    "deep_dive_summary",
    "deep_dive_recommendations",
    "deep_dive_next_events",
    "deep_dive_previous_events",
]

loaded = {}

for name in required_tables:
    loaded[name] = load_processed_table(name)
    print(name, loaded[name].shape)

event_metrics = loaded["metrics_event_level"]
family_metrics = loaded["metrics_family_level"]
depth_distribution = loaded["metrics_repeat_depth"]
retention_by_first_family = loaded["metrics_first_event_retention"]
event_transitions = loaded["event_transitions"]
family_transitions = loaded["family_transitions"]
event_scorecard = loaded["event_scorecard"]
deep_dive_summary = loaded["deep_dive_summary"]
deep_dive_recommendations = loaded["deep_dive_recommendations"]
deep_dive_next_events = loaded["deep_dive_next_events"]
deep_dive_previous_events = loaded["deep_dive_previous_events"]


metrics_event_level (18, 29)
metrics_family_level (13, 12)
metrics_repeat_depth (5, 3)
metrics_first_event_retention (11, 5)
event_transitions (64, 3)
family_transitions (58, 3)
event_scorecard (18, 15)
deep_dive_summary (6, 33)
deep_dive_recommendations (6, 11)
deep_dive_next_events (11, 3)
deep_dive_previous_events (37, 3)


In [7]:
print("events:", event_metrics["event_name"].nunique())
print("families:", family_metrics["event_family"].nunique())
print("deep dive events:", deep_dive_summary["event_name"].nunique())

display(event_metrics.head())
display(family_metrics.head())
display(deep_dive_summary.head())


events: 18
families: 13
deep dive events: 6


,canonical_event_id,event_name,event_family,event_year,event_date,event_season,event_type,is_paid,approx_capacity,target_audience,format,strategic_role,notes,event_season_order,metadata_order,event_order,raw_rows,full_unique_participants,rows_with_any_identifier,rows_with_strong_identifier,clean_unique_participants,strong_identifier_share,clean_coverage_share,clean_participants,new_participants,repeat_participants,avg_prior_events,new_share,repeat_share
0,posvyat_2023,Посвят'23,Посвят,2023,None,autumn,onboarding,None,300-500,first-year students,overnight trip,acquisition,Посвящение в студенты; выезд с ночевкой в Подмосковье; квесты и вечерняя программа,4,0,202304.000,354,354,354,354,287,1.000000,0.810734,287,287,0,0.000000,1.000,0.000
1,antiposvyat_2023,Антипосвят'23,Антипосвят,2023,None,autumn,party,None,150-200,senior students,offline party,retention/community,Мероприятие от первокурсников для старшекурсников,4,1,202304.001,154,153,154,154,82,1.000000,0.535948,82,72,10,0.121951,0.878,0.122
2,ball_fkn_2024,Бал ФКН'24,Бал ФКН,2024,None,spring,party,None,unknown,HSE FCS students,offline party,community,Весенний бал ФКН,2,2,202402.002,705,699,703,0,0,0.000000,0.000000,0,0,0,0.000000,0.000,0.000
3,ballmer_peak_2024,Ballmer Peak'24,Ballmer Peak,2024,None,spring,hackathon,None,unknown,students,mini hackathon,engagement,Мини-хахатон,2,4,202402.004,150,143,146,146,109,0.973333,0.762238,109,84,25,0.229358,0.771,0.229
4,neyrorave_2024,Нейрорейв,Нейрорейв,2024,None,spring,party,FALSE,300,students,offline rave,acquisition/reach,Бесплатная тусовка-коллаборация с ФКН; рейв и светомузыка,2,7,202402.007,859,809,857,794,728,0.924331,0.899876,728,728,0,0.000000,1.000,0.000


,event_family,events,raw_rows,full_unique_participants,clean_unique_participants,clean_participants,new_participants,repeat_participants,avg_strong_identifier_share,avg_clean_coverage_share,new_share,repeat_share
0,Нейрорейв,1,859,809,728,728,728,0,0.924331,0.899876,1.000,0.000
1,Посвят,2,771,771,687,687,684,3,1.000000,0.884984,0.996,0.004
2,Поход,2,602,564,443,443,336,107,0.998413,0.786545,0.758,0.242
3,CSFEST,1,371,356,259,259,214,45,1.000000,0.727528,0.826,0.174
4,Экватор,2,561,506,200,200,161,39,0.891029,0.392390,0.805,0.195


,canonical_event_id,event_name,event_family,event_year,event_date,event_season,event_type,is_paid,approx_capacity,target_audience,format,strategic_role,notes,event_season_order,metadata_order,event_order,raw_rows,full_unique_participants,rows_with_any_identifier,rows_with_strong_identifier,clean_unique_participants,strong_identifier_share,clean_coverage_share,clean_participants,new_participants,repeat_participants,avg_prior_events,new_share,repeat_share,participants_with_future_event,future_event_rate,product_role,data_quality_status
0,neyrorave_2024,Нейрорейв,Нейрорейв,2024,None,spring,party,FALSE,300,students,offline rave,acquisition/reach,Бесплатная тусовка-коллаборация с ФКН; рейв и светомузыка,2,7,202402.007,859,809,857,794,728,0.924331,0.899876,728,728,0,0.000000,1.000,0.000,1,0.001,acquisition engine,good
1,posvyat_2025,Посвят'25,Посвят,2025,None,autumn,onboarding,None,300-500,first-year students,overnight trip,acquisition,Главный onboarding-event для первокурсников,4,12,202504.012,417,417,417,417,400,1.000000,0.959233,400,397,3,0.010000,0.992,0.008,12,0.030,acquisition engine,good
2,csfest_2025,CSFEST'25,CSFEST,2025,None,spring,university event,None,500,students/teachers,offline show,reach/community,Коллаборация студентов и преподавателей; проводится в апреле,2,14,202502.014,371,356,371,371,259,1.000000,0.727528,259,214,45,0.235521,0.826,0.174,38,0.147,reach event,medium
3,pohod_2025,Поход'25,Поход,2025,None,summer,outdoor trip,None,200,students/community,camping trip,retention/community,Поход с палатками и активностями,3,15,202503.015,315,291,315,314,219,0.996825,0.752577,219,129,90,0.543379,0.589,0.411,11,0.050,retention/community engine,good
4,antiposvyat_2025,Антипосвят'25,Антипосвят,2025,None,autumn,party,None,150-200,senior students,offline party,retention/community,Attendance в данных занижен; нужна корректировка в attendance_corrections.csv,4,13,202504.013,221,220,220,220,113,0.995475,0.513636,113,66,47,0.628319,0.584,0.416,0,0.000,retention/community engine,medium


In [8]:
total_events = event_metrics["event_name"].nunique()

events_with_clean = int((event_metrics["clean_participants"] > 0).sum())

total_clean_participant_event_pairs = int(event_metrics["clean_participants"].sum())

largest_event = (
    event_metrics
    .sort_values("clean_participants", ascending=False)
    .iloc[0]
)

top_repeat_event = (
    event_metrics[event_metrics["clean_participants"] > 0]
    .sort_values("repeat_share", ascending=False)
    .iloc[0]
)

top_scorecard_event = event_scorecard.iloc[0]

repeat_depth_1_share = float(
    depth_distribution.loc[
        depth_distribution["events_cnt"] == 1,
        "participant_share"
    ].iloc[0]
)

multi_event_share = round(1 - repeat_depth_1_share, 3)

executive_kpis = pd.DataFrame([
    {
        "metric": "events_total",
        "value": total_events,
        "interpretation": "Всего событий в event portfolio",
    },
    {
        "metric": "events_with_clean_identity",
        "value": events_with_clean,
        "interpretation": "Событий с участниками в clean identity layer",
    },
    {
        "metric": "clean_participant_event_pairs",
        "value": total_clean_participant_event_pairs,
        "interpretation": "Суммарные event-participant пары в clean layer",
    },
    {
        "metric": "largest_clean_event",
        "value": largest_event["event_name"],
        "interpretation": f"{int(largest_event['clean_participants'])} clean participants",
    },
    {
        "metric": "highest_repeat_share_event",
        "value": top_repeat_event["event_name"],
        "interpretation": f"repeat_share = {top_repeat_event['repeat_share']:.3f}",
    },
    {
        "metric": "multi_event_participant_share",
        "value": multi_event_share,
        "interpretation": "Доля участников, посетивших 2+ событий",
    },
    {
        "metric": "top_portfolio_score_event",
        "value": top_scorecard_event["event_name"],
        "interpretation": f"portfolio_score = {top_scorecard_event['portfolio_score']:.3f}",
    },
])

display(executive_kpis)


,metric,value,interpretation
0,events_total,18,Всего событий в event portfolio
1,events_with_clean_identity,16,Событий с участниками в clean identity layer
2,clean_participant_event_pairs,2853,Суммарные event-participant пары в clean layer
3,largest_clean_event,Нейрорейв,728 clean participants
4,highest_repeat_share_event,Настолки'24,repeat_share = 0.511
5,multi_event_participant_share,0.097,"Доля участников, посетивших 2+ событий"
6,top_portfolio_score_event,CSFEST'25,portfolio_score = 0.728


In [9]:
acquisition_events = (
    event_metrics[
        event_metrics["clean_participants"] > 0
    ]
    .sort_values(["new_participants", "new_share"], ascending=False)
    [
        [
            "event_name",
            "event_family",
            "clean_participants",
            "new_participants",
            "new_share",
            "repeat_share",
            "clean_coverage_share",
        ]
    ]
)

display(acquisition_events.head(10))


,event_name,event_family,clean_participants,new_participants,new_share,repeat_share,clean_coverage_share
4,Нейрорейв,Нейрорейв,728,728,1.000,0.000,0.899876
15,Посвят'25,Посвят,400,397,0.992,0.008,0.959233
0,Посвят'23,Посвят,287,287,1.000,0.000,0.810734
12,CSFEST'25,CSFEST,259,214,0.826,0.174,0.727528
5,Поход'24,Поход,224,207,0.924,0.076,0.820513
13,Поход'25,Поход,219,129,0.589,0.411,0.752577
6,Экватор'24,Экватор,121,98,0.810,0.190,0.458333
3,Ballmer Peak'24,Ballmer Peak,109,84,0.771,0.229,0.762238
1,Антипосвят'23,Антипосвят,82,72,0.878,0.122,0.535948
16,Антипосвят'25,Антипосвят,113,66,0.584,0.416,0.513636


In [10]:
retention_events = (
    event_metrics[
        event_metrics["clean_participants"] >= 25
    ]
    .sort_values(["repeat_share", "repeat_participants"], ascending=False)
    [
        [
            "event_name",
            "event_family",
            "clean_participants",
            "repeat_participants",
            "repeat_share",
            "new_share",
            "clean_coverage_share",
        ]
    ]
)

display(retention_events.head(10))


,event_name,event_family,clean_participants,repeat_participants,repeat_share,new_share,clean_coverage_share
10,Настолки'24,Настолки,47,24,0.511,0.489,0.522222
16,Антипосвят'25,Антипосвят,113,47,0.416,0.584,0.513636
13,Поход'25,Поход,219,90,0.411,0.589,0.752577
11,ЗВ'25,Зимний выезд,30,7,0.233,0.767,0.517241
3,Ballmer Peak'24,Ballmer Peak,109,25,0.229,0.771,0.762238
14,Экватор'25,Экватор,79,16,0.203,0.797,0.326446
6,Экватор'24,Экватор,121,23,0.190,0.810,0.458333
12,CSFEST'25,CSFEST,259,45,0.174,0.826,0.727528
7,GLANZ,GLANZ,73,11,0.151,0.849,0.439759
8,ANNIVERSARY'24,Anniversary,53,8,0.151,0.849,0.654321


In [11]:
top_event_transitions = event_transitions.head(15)
top_family_transitions = family_transitions.head(15)

display(top_event_transitions)
display(top_family_transitions)


,event_name,next_event_name,participants
0,Поход'24,Поход'25,50
1,CSFEST'25,Поход'25,27
2,Посвят'23,Ballmer Peak'24,19
3,Посвят'25,Антипосвят'25,12
4,Посвят'23,CSFEST'25,10
5,Ballmer Peak'24,Экватор'24,10
6,Посвят'23,Антипосвят'23,10
7,Посвят'23,Поход'24,10
8,Поход'24,CSFEST'25,9
9,Посвят'23,Настолки'24,9


,event_family,next_event_family,participants
0,Поход,Поход,50
1,CSFEST,Поход,27
2,Посвят,Антипосвят,24
3,Посвят,Ballmer Peak,19
4,Ballmer Peak,Экватор,13
5,Посвят,Поход,11
6,Посвят,CSFEST,10
7,Посвят,Настолки,9
8,Поход,Антипосвят,9
9,Поход,CSFEST,9


In [12]:
data_quality_events = (
    event_metrics
    .copy()
    .sort_values("clean_coverage_share", ascending=True)
    [
        [
            "event_name",
            "event_family",
            "raw_rows",
            "full_unique_participants",
            "clean_unique_participants",
            "strong_identifier_share",
            "clean_coverage_share",
        ]
    ]
)

display(data_quality_events)


,event_name,event_family,raw_rows,full_unique_participants,clean_unique_participants,strong_identifier_share,clean_coverage_share
2,Бал ФКН'24,Бал ФКН,705,699,0,0.000000,0.000000
14,Экватор'25,Экватор,245,242,79,0.987755,0.326446
17,ЗВ'26,Зимний выезд,69,68,29,0.927536,0.426471
7,GLANZ,GLANZ,281,166,73,0.583630,0.439759
6,Экватор'24,Экватор,316,264,121,0.794304,0.458333
16,Антипосвят'25,Антипосвят,221,220,113,0.995475,0.513636
11,ЗВ'25,Зимний выезд,60,58,30,1.000000,0.517241
10,Настолки'24,Настолки,91,90,47,1.000000,0.522222
1,Антипосвят'23,Антипосвят,154,153,82,1.000000,0.535948
8,ANNIVERSARY'24,Anniversary,86,81,53,0.965116,0.654321


In [13]:
product_findings = pd.DataFrame([
    {
        "theme": "Growth / acquisition",
        "finding": "Массовые события дают основной приток новой аудитории.",
        "evidence": "В clean layer крупнейшими acquisition/reach событиями являются Нейрорейв, Посвят'25 и Посвят'23.",
        "business_meaning": "Эти события нужно рассматривать как верх воронки event-community product.",
    },
    {
        "theme": "Retention / community",
        "finding": "Самый сильный repeat-сигнал дают более камерные или community-heavy события.",
        "evidence": "Настолки'24, Антипосвят'25 и Поход'25 имеют самые высокие repeat_share среди событий с участниками.",
        "business_meaning": "Для удержания важен не только размер события, но и формат, создающий повторное вовлечение.",
    },
    {
        "theme": "Journeys",
        "finding": "Есть заметные переходы между событиями одной семьи и соседними community-форматами.",
        "evidence": "Сильный переход Поход'24 → Поход'25, а также переходы из CSFEST'25 и Посвят'23 в следующие события.",
        "business_meaning": "Коммуникации после события можно строить вокруг следующего логичного шага пользователя.",
    },
    {
        "theme": "Portfolio strategy",
        "finding": "Портфель событий работает как двухконтурная система.",
        "evidence": "Нейрорейв и Посвят дают acquisition, а Поход, Антипосвят и Настолки дают retention/community-сигнал.",
        "business_meaning": "События нельзя сравнивать только по размеру аудитории: у разных форматов разные продуктовые роли.",
    },
    {
        "theme": "Data quality",
        "finding": "Часть событий требует улучшения сбора идентификаторов.",
        "evidence": "Бал ФКН'24 и Коллаб'24 не попали в clean retention metrics из-за слабого или нестандартного identity layer.",
        "business_meaning": "Для будущей аналитики нужно стандартизировать registration forms и обязательные поля.",
    },
])

display(product_findings)


,theme,finding,evidence,business_meaning
0,Growth / acquisition,Массовые события дают основной приток новой аудитории.,"В clean layer крупнейшими acquisition/reach событиями являются Нейрорейв, Посвят'25 и Посвят'23.",Эти события нужно рассматривать как верх воронки event-community product.
1,Retention / community,Самый сильный repeat-сигнал дают более камерные или community-heavy события.,"Настолки'24, Антипосвят'25 и Поход'25 имеют самые высокие repeat_share среди событий с участниками.","Для удержания важен не только размер события, но и формат, создающий повторное вовлечение."
2,Journeys,Есть заметные переходы между событиями одной семьи и соседними community-форматами.,"Сильный переход Поход'24 → Поход'25, а также переходы из CSFEST'25 и Посвят'23 в следующие события.",Коммуникации после события можно строить вокруг следующего логичного шага пользователя.
3,Portfolio strategy,Портфель событий работает как двухконтурная система.,"Нейрорейв и Посвят дают acquisition, а Поход, Антипосвят и Настолки дают retention/community-сигнал.",События нельзя сравнивать только по размеру аудитории: у разных форматов разные продуктовые роли.
4,Data quality,Часть событий требует улучшения сбора идентификаторов.,Бал ФКН'24 и Коллаб'24 не попали в clean retention metrics из-за слабого или нестандартного identity layer.,Для будущей аналитики нужно стандартизировать registration forms и обязательные поля.


In [14]:
actionable_recommendations = pd.DataFrame([
    {
        "priority": "P1",
        "area": "Post-event activation",
        "recommendation": "После крупных acquisition-событий запускать follow-up цепочку с приглашением на ближайшее community-событие.",
        "why": "Нейрорейв и Посвят привлекают много новых участников, но без follow-up часть аудитории не переходит в повторное участие.",
        "expected_effect": "Рост repeat participation и доли участников, посетивших 2+ событий.",
        "metric_to_track": "future_event_rate, repeat_share, 2+ events share",
    },
    {
        "priority": "P1",
        "area": "Retention mechanics",
        "recommendation": "Использовать Поход, Антипосвят и Настолки как retention/community-ядро портфеля.",
        "why": "Эти форматы показывают высокий repeat_share и лучше отражают качество вовлечения.",
        "expected_effect": "Укрепление ядра комьюнити и рост возвращаемости.",
        "metric_to_track": "repeat_share, returned_later, event_depth",
    },
    {
        "priority": "P1",
        "area": "Registration data quality",
        "recommendation": "Стандартизировать регистрационные формы: обязательные Telegram или email, единый формат ФИО, единые поля курса/программы.",
        "why": "Без стабильных идентификаторов невозможно надёжно считать retention и journeys.",
        "expected_effect": "Рост clean identity coverage и снижение identity conflicts.",
        "metric_to_track": "clean_coverage_share, strong_identifier_share, identity_conflict_rate",
    },
    {
        "priority": "P2",
        "area": "Event journey design",
        "recommendation": "Проектировать события как цепочки: например, onboarding → community event → seasonal event.",
        "why": "В данных уже видны переходы между событиями, например Поход'24 → Поход'25.",
        "expected_effect": "Более управляемое развитие participant journey.",
        "metric_to_track": "event_transition_rate, family_transition_rate",
    },
    {
        "priority": "P2",
        "area": "Deep-dive data recovery",
        "recommendation": "Сделать special loaders для Коллаб'24 и Бал ФКН'24.",
        "why": "Сейчас эти события плохо представлены в clean metrics, хотя потенциально важны для портфеля.",
        "expected_effect": "Более полное покрытие портфеля и честное сравнение событий.",
        "metric_to_track": "clean_participants recovered, coverage_share",
    },
])

display(actionable_recommendations)


,priority,area,recommendation,why,expected_effect,metric_to_track
0,P1,Post-event activation,После крупных acquisition-событий запускать follow-up цепочку с приглашением на ближайшее community-событие.,"Нейрорейв и Посвят привлекают много новых участников, но без follow-up часть аудитории не переходит в повторное участие.","Рост repeat participation и доли участников, посетивших 2+ событий.","future_event_rate, repeat_share, 2+ events share"
1,P1,Retention mechanics,"Использовать Поход, Антипосвят и Настолки как retention/community-ядро портфеля.",Эти форматы показывают высокий repeat_share и лучше отражают качество вовлечения.,Укрепление ядра комьюнити и рост возвращаемости.,"repeat_share, returned_later, event_depth"
2,P1,Registration data quality,"Стандартизировать регистрационные формы: обязательные Telegram или email, единый формат ФИО, единые поля курса/программы.",Без стабильных идентификаторов невозможно надёжно считать retention и journeys.,Рост clean identity coverage и снижение identity conflicts.,"clean_coverage_share, strong_identifier_share, identity_conflict_rate"
3,P2,Event journey design,"Проектировать события как цепочки: например, onboarding → community event → seasonal event.","В данных уже видны переходы между событиями, например Поход'24 → Поход'25.",Более управляемое развитие participant journey.,"event_transition_rate, family_transition_rate"
4,P2,Deep-dive data recovery,Сделать special loaders для Коллаб'24 и Бал ФКН'24.,"Сейчас эти события плохо представлены в clean metrics, хотя потенциально важны для портфеля.",Более полное покрытие портфеля и честное сравнение событий.,"clean_participants recovered, coverage_share"


In [15]:
readme_summary = f"""
# Ключевые выводы

Проект рассматривает портфель мероприятий cstati как event-community product: разные события играют разные роли в росте, удержании и развитии комьюнити.

## Основные цифры

- Всего событий в анализе: **{total_events}**
- Событий с clean identity coverage: **{events_with_clean}**
- Clean event-participant pairs: **{total_clean_participant_event_pairs}**
- Крупнейшее событие в clean layer: **{largest_event['event_name']}** ({int(largest_event['clean_participants'])} участников)
- Доля участников, посетивших 2+ событий: **{multi_event_share:.1%}**
- Топ event по portfolio score: **{top_scorecard_event['event_name']}**

## Главный продуктовый вывод

Портфель мероприятий работает как двухконтурная система:

1. **Acquisition-контур** — массовые события привлекают новую аудиторию.
2. **Retention/community-контур** — более камерные и регулярные события возвращают участников и формируют ядро комьюнити.

## Рекомендации

1. Запустить post-event activation после крупных acquisition-событий.
2. Использовать Поход, Антипосвят и Настолки как retention/community-ядро.
3. Стандартизировать регистрационные формы для улучшения clean identity coverage.
4. Проектировать события как цепочки, а не как независимые мероприятия.
5. Добавить special loaders для событий со сложной структурой данных.
"""

print(readme_summary)



# Ключевые выводы

Проект рассматривает портфель мероприятий cstati как event-community product: разные события играют разные роли в росте, удержании и развитии комьюнити.

## Основные цифры

- Всего событий в анализе: **18**
- Событий с clean identity coverage: **16**
- Clean event-participant pairs: **2853**
- Крупнейшее событие в clean layer: **Нейрорейв** (728 участников)
- Доля участников, посетивших 2+ событий: **9.7%**
- Топ event по portfolio score: **CSFEST'25**

## Главный продуктовый вывод

Портфель мероприятий работает как двухконтурная система:

1. **Acquisition-контур** — массовые события привлекают новую аудиторию.
2. **Retention/community-контур** — более камерные и регулярные события возвращают участников и формируют ядро комьюнити.

## Рекомендации

1. Запустить post-event activation после крупных acquisition-событий.
2. Использовать Поход, Антипосвят и Настолки как retention/community-ядро.
3. Стандартизировать регистрационные формы для улучшения clean identity cove

In [16]:
final_outputs = {
    "final_executive_kpis": executive_kpis,
    "final_product_findings": product_findings,
    "final_actionable_recommendations": actionable_recommendations,
    "final_acquisition_events": acquisition_events,
    "final_retention_events": retention_events,
    "final_data_quality_events": data_quality_events,
}

for name, df in final_outputs.items():
    output_path = PROCESSED_PUBLIC_DIR / f"{name}.csv"
    df.to_csv(output_path, index=False)
    print("saved:", output_path)

readme_summary_path = DOCS_DIR / "readme_summary.md"
readme_summary_path.write_text(readme_summary, encoding="utf-8")

print("saved:", readme_summary_path)


saved: /content/data/processed_public/final_executive_kpis.csv
saved: /content/data/processed_public/final_product_findings.csv
saved: /content/data/processed_public/final_actionable_recommendations.csv
saved: /content/data/processed_public/final_acquisition_events.csv
saved: /content/data/processed_public/final_retention_events.csv
saved: /content/data/processed_public/final_data_quality_events.csv
saved: /content/docs/readme_summary.md


In [17]:
print("=== EXECUTIVE KPIS ===")
display(executive_kpis)

print("\n=== PRODUCT FINDINGS ===")
display(product_findings)

print("\n=== ACTIONABLE RECOMMENDATIONS ===")
display(actionable_recommendations)

print("\n=== README SUMMARY ===")
print(readme_summary)



=== EXECUTIVE KPIS ===


,metric,value,interpretation
0,events_total,18,Всего событий в event portfolio
1,events_with_clean_identity,16,Событий с участниками в clean identity layer
2,clean_participant_event_pairs,2853,Суммарные event-participant пары в clean layer
3,largest_clean_event,Нейрорейв,728 clean participants
4,highest_repeat_share_event,Настолки'24,repeat_share = 0.511
5,multi_event_participant_share,0.097,"Доля участников, посетивших 2+ событий"
6,top_portfolio_score_event,CSFEST'25,portfolio_score = 0.728



=== PRODUCT FINDINGS ===


,theme,finding,evidence,business_meaning
0,Growth / acquisition,Массовые события дают основной приток новой аудитории.,"В clean layer крупнейшими acquisition/reach событиями являются Нейрорейв, Посвят'25 и Посвят'23.",Эти события нужно рассматривать как верх воронки event-community product.
1,Retention / community,Самый сильный repeat-сигнал дают более камерные или community-heavy события.,"Настолки'24, Антипосвят'25 и Поход'25 имеют самые высокие repeat_share среди событий с участниками.","Для удержания важен не только размер события, но и формат, создающий повторное вовлечение."
2,Journeys,Есть заметные переходы между событиями одной семьи и соседними community-форматами.,"Сильный переход Поход'24 → Поход'25, а также переходы из CSFEST'25 и Посвят'23 в следующие события.",Коммуникации после события можно строить вокруг следующего логичного шага пользователя.
3,Portfolio strategy,Портфель событий работает как двухконтурная система.,"Нейрорейв и Посвят дают acquisition, а Поход, Антипосвят и Настолки дают retention/community-сигнал.",События нельзя сравнивать только по размеру аудитории: у разных форматов разные продуктовые роли.
4,Data quality,Часть событий требует улучшения сбора идентификаторов.,Бал ФКН'24 и Коллаб'24 не попали в clean retention metrics из-за слабого или нестандартного identity layer.,Для будущей аналитики нужно стандартизировать registration forms и обязательные поля.



=== ACTIONABLE RECOMMENDATIONS ===


,priority,area,recommendation,why,expected_effect,metric_to_track
0,P1,Post-event activation,После крупных acquisition-событий запускать follow-up цепочку с приглашением на ближайшее community-событие.,"Нейрорейв и Посвят привлекают много новых участников, но без follow-up часть аудитории не переходит в повторное участие.","Рост repeat participation и доли участников, посетивших 2+ событий.","future_event_rate, repeat_share, 2+ events share"
1,P1,Retention mechanics,"Использовать Поход, Антипосвят и Настолки как retention/community-ядро портфеля.",Эти форматы показывают высокий repeat_share и лучше отражают качество вовлечения.,Укрепление ядра комьюнити и рост возвращаемости.,"repeat_share, returned_later, event_depth"
2,P1,Registration data quality,"Стандартизировать регистрационные формы: обязательные Telegram или email, единый формат ФИО, единые поля курса/программы.",Без стабильных идентификаторов невозможно надёжно считать retention и journeys.,Рост clean identity coverage и снижение identity conflicts.,"clean_coverage_share, strong_identifier_share, identity_conflict_rate"
3,P2,Event journey design,"Проектировать события как цепочки: например, onboarding → community event → seasonal event.","В данных уже видны переходы между событиями, например Поход'24 → Поход'25.",Более управляемое развитие participant journey.,"event_transition_rate, family_transition_rate"
4,P2,Deep-dive data recovery,Сделать special loaders для Коллаб'24 и Бал ФКН'24.,"Сейчас эти события плохо представлены в clean metrics, хотя потенциально важны для портфеля.",Более полное покрытие портфеля и честное сравнение событий.,"clean_participants recovered, coverage_share"



=== README SUMMARY ===

# Ключевые выводы

Проект рассматривает портфель мероприятий cstati как event-community product: разные события играют разные роли в росте, удержании и развитии комьюнити.

## Основные цифры

- Всего событий в анализе: **18**
- Событий с clean identity coverage: **16**
- Clean event-participant pairs: **2853**
- Крупнейшее событие в clean layer: **Нейрорейв** (728 участников)
- Доля участников, посетивших 2+ событий: **9.7%**
- Топ event по portfolio score: **CSFEST'25**

## Главный продуктовый вывод

Портфель мероприятий работает как двухконтурная система:

1. **Acquisition-контур** — массовые события привлекают новую аудиторию.
2. **Retention/community-контур** — более камерные и регулярные события возвращают участников и формируют ядро комьюнити.

## Рекомендации

1. Запустить post-event activation после крупных acquisition-событий.
2. Использовать Поход, Антипосвят и Настолки как retention/community-ядро.
3. Стандартизировать регистрационные формы для улучш

## Финальные итоги

В этом ноутбуке собраны финальные продуктовые выводы и рекомендации.

### Главный вывод

cstati event portfolio работает как event-community product, где разные события выполняют разные роли:

- массовые события отвечают за acquisition и reach;
- community-heavy события отвечают за retention и повторное вовлечение;
- event journeys можно проектировать и усиливать через follow-up коммуникации.

### Практические рекомендации

1. Строить post-event activation после крупных событий.
2. Развивать retention-механики в Походах, Антипосвятах и Настолках.
3. Стандартизировать регистрационные формы.
4. Использовать event journeys для планирования коммуникаций.
5. Улучшить ETL для событий со сложной структурой.

### Что пойдёт в README

- executive summary;
- 3–5 ключевых метрик;
- portfolio health interpretation;
- event roles;
- финальные рекомендации;
- privacy и methodology notes.
